# 11 · Sparrow Engine as an HTTP server

The other notebooks call the engine inside the Python process. Here the engine runs as a separate
service (the `sparrow-engine-server` Docker image) and you talk to it over HTTP. Use this when the
code that needs predictions is not Python, runs on another machine, or when several programs should
share one loaded model.

**Needs Docker** on the machine that runs this notebook (it will not work on Colab). If a server is
already running somewhere, set `SPP_SERVER_URL` (for example `http://gpu-box:8080`) and the notebook
uses it instead of starting a container.

| Route | Does |
|---|---|
| `GET /healthz`, `GET /v1/health` | liveness, readiness and engine version |
| `GET /v1/catalog` | models the server found in its model folder |
| `POST /v1/detect?model=…` | detect on one image (multipart field `image`) |
| `POST /v1/detect/batch?model=…` | detect on several images (repeated field `images`) |
| `POST /v1/classify?model=…&top_k=…` | classify a whole image |
| `POST /v1/pipeline?detector=…&classifier=…` | detect, then classify each box |

Boxes come back normalized to 0–1 (`x_min`, `y_min`, `x_max`, `y_max`).

In [ ]:
# Setup: the server does the inference; this notebook only needs an HTTP client.
# sparrow-engine is used for one thing: downloading models into the folder the server reads.
import importlib.util
import subprocess
import sys

for pkg, mod in [("sparrow-engine>=0.1.31", "sparrow_engine"), ("requests", "requests"),
                 ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("pillow", "PIL")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import sparrow_engine as spe
import requests

print("sparrow-engine", spe.__version__)

In [ ]:
# Demo data: downloaded once into ./data and checked against a published checksum.
import hashlib
import os
import shutil
import urllib.request
import zipfile
from pathlib import Path

DATA = Path(os.environ.get("SPP_DATA_DIR", "data")).resolve()
ASSETS = {
    # Pytorch-Wildlife demo assets, CC-BY-4.0, https://zenodo.org/records/15376499
    "demo_data_det.zip": (
        "https://zenodo.org/records/15376499/files/demo_data_det.zip?download=1",
        "md5", "373e15c8e7c8894379a82086a9177763",
    ),
    "demo_data_base.zip": (
        "https://zenodo.org/records/15376499/files/demo_data_base.zip?download=1",
        "md5", "f89b9b76c753d90e4296d62d71cf1732",
    ),
    # "Yellow-Throated Vireo Song" by SwampVids, CC-BY-4.0, Wikimedia Commons
    "vireo.wav": (
        "https://upload.wikimedia.org/wikipedia/commons/4/4f/"
        "Vireo_flavifrons_-_Yellow-Throated_Vireo_Song.wav",
        "sha1", "2b472209c1b18bfc0c225975838e3a34c1e9887a",
    ),
}


def fetch(name):
    """Download an asset (once), verify it, and unzip archives into DATA."""
    url, algo, digest = ASSETS[name]
    target = DATA / name
    DATA.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        print(f"downloading {name} ...")
        part = target.with_name(target.name + ".part")
        req = urllib.request.Request(url, headers={"User-Agent": "sparrow-playground"})
        with urllib.request.urlopen(req) as r, open(part, "wb") as f:
            shutil.copyfileobj(r, f)
        part.rename(target)
    h = hashlib.new(algo)
    with open(target, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    if h.hexdigest() != digest:
        target.unlink()
        raise RuntimeError(f"{name}: checksum mismatch, deleted; run the cell again")
    marker = DATA / f".{name}.extracted"
    if name.endswith(".zip") and not marker.exists():
        with zipfile.ZipFile(target) as z:
            z.extractall(DATA)
        marker.touch()
    return target

In [ ]:
fetch("demo_data_det.zip")
images = sorted((DATA / "data/data_example/images").rglob("*.jpg"))
print(len(images), "images")

# The server reads models from the folder mounted at /models. Download what we need into it first:
# the server finds models when it starts.
MODELS = DATA / "server_models"
DETECTOR, CLASSIFIER = "MDV6-yolov10-c", "AI4G-Amazon-V2"
spe.download_models([DETECTOR, CLASSIFIER], dest=MODELS)
print(sorted(p.name for p in MODELS.iterdir() if p.is_dir()))

## Start the server

This is the same as running, in a terminal:

```bash
docker run -d --rm --name spp-engine-server -p 8080:8080 \
    -v "$PWD/data/server_models:/models:ro" \
    zhongqimiao/sparrow-engine-server:v0.1.32
```

Use the `sparrow-engine-server-gpu` image with `--gpus all` on an NVIDIA machine.

In [ ]:
import os
import shutil
import time

IMAGE = os.environ.get("SPP_SERVER_IMAGE", "zhongqimiao/sparrow-engine-server:v0.1.32")
PORT = int(os.environ.get("SPP_SERVER_PORT", "8080"))
CONTAINER = "spp-engine-server"
URL = os.environ.get("SPP_SERVER_URL")
started_here = False

if URL is None:
    if shutil.which("docker") is None:
        raise RuntimeError("Docker is not installed. Install it, or set SPP_SERVER_URL to a running server.")
    subprocess.run(["docker", "rm", "-f", CONTAINER], capture_output=True)
    subprocess.check_call([
        "docker", "run", "-d", "--rm", "--name", CONTAINER,
        "-p", f"127.0.0.1:{PORT}:8080",
        "-v", f"{MODELS}:/models:ro",
        IMAGE,
    ], stdout=subprocess.DEVNULL)
    started_here = True
    URL = f"http://localhost:{PORT}"

for _ in range(120):  # the first run pulls the image, then startup takes a few seconds
    try:
        if requests.get(f"{URL}/healthz", timeout=2).ok:
            break
    except requests.ConnectionError:
        pass
    time.sleep(1)
else:
    raise RuntimeError(f"server at {URL} did not come up; try `docker logs {CONTAINER}`")

print(requests.get(f"{URL}/v1/health").json())

In [ ]:
import pandas as pd

pd.DataFrame(requests.get(f"{URL}/v1/catalog").json())[["model_id", "model_type", "family", "loaded"]]

## Detect on one image

From a shell the same request is

```bash
curl -F image=@photo.jpg "http://localhost:8080/v1/detect?model=MDV6-yolov10-c&threshold=0.2"
```

Any language with an HTTP client can do the same.

In [ ]:
def post(route, files, **params):
    r = requests.post(f"{URL}{route}", params=params, files=files, timeout=120)
    r.raise_for_status()
    return r.json()

def image_field(path, name="image"):
    return (name, (path.name, path.read_bytes(), "image/jpeg"))

res = post("/v1/detect", [image_field(images[1])], model=DETECTOR, threshold=0.2)
res

## Batch detect

One request for many images; `index` in each result is the position in the request.

In [ ]:
batch = post("/v1/detect/batch", [image_field(p, "images") for p in images], model=DETECTOR, threshold=0.2)

rows = []
for r in batch["results"]:
    for d in r["detections"]:
        rows.append({"file": images[r["index"]].name, "label": d["label"],
                     "confidence": round(d["confidence"], 3), **d["bbox"]})
df = pd.DataFrame(rows)
print(f"{batch['count']} images in {batch['processing_time_ms']:.0f} ms, {len(df)} detections")
df.head(10)

## Detect, then classify each animal

`/v1/pipeline` runs the detector, crops each box and runs the classifier on the crop.

In [ ]:
res = post("/v1/pipeline", [image_field(images[1])], detector=DETECTOR, classifier=CLASSIFIER, top_k=3)
pd.DataFrame([{"detector": d["label"], "det_conf": round(d["confidence"], 3),
               "species": d["classification"]["label"],
               "species_conf": round(d["classification"]["confidence"], 3)}
              for d in res["detections"]])

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

img = Image.open(images[1])
W, H = img.size
fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(img)
for d in res["detections"]:
    b = d["bbox"]
    x0, y0, x1, y1 = b["x_min"] * W, b["y_min"] * H, b["x_max"] * W, b["y_max"] * H
    ax.add_patch(patches.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, color="lime", lw=2))
    c = d["classification"]
    ax.text(x0, y0 - 4, f"{c['label']} {c['confidence']:.2f}", color="lime", fontsize=11)
ax.axis("off")
plt.show()

## Stop the server

In [ ]:
if started_here:
    subprocess.run(["docker", "stop", CONTAINER], capture_output=True)
    print("stopped", CONTAINER)

## Next

- The server also has `/v1/audio/detect`, `/v1/audio/events`, `/v1/embed` and `/v1/embed/batch`. The full API is in the engine's [HTTP integration guide](https://github.com/microsoft/SPARROW-Engine/blob/main/docs/integration/web.md).
- Calls that change server state (`/v1/models`, `/v1/models/load`) need `SPARROW_ENGINE_MANAGEMENT_TOKEN` set on the server.
- Other ways to run the same models: [`interfaces/cli`](../interfaces/cli) (the `spe` command) and [`interfaces/c`](../interfaces/c) (the C library).